### Analisis Hasil Simulasi pada Temperatur $T = 4,0$

Berdasarkan hasil lima kali pengulangan simulasi pada temperatur $T = 4,0$, diperoleh pola yang relatif konsisten meskipun konfigurasi spin akhir pada setiap simulasi berbeda. Perbedaan konfigurasi tersebut disebabkan oleh sifat stokastik algoritma Metropolis Monte Carlo yang menggunakan pemilihan spin dan bilangan acak dalam proses pembaruan konfigurasi. Akibatnya, setiap simulasi menghasilkan susunan spin akhir yang berbeda walaupun menggunakan parameter masukan yang sama.

![Hasil Simulasi 1 T=4](./Hasil/TInggi/G20T4.0R1_20260701_093400.png)
![Hasil Simulasi 2 T=4](./Hasil/TInggi/G20T4.0R2_20260701_093407.png)
![Hasil Simulasi 3 T=4](./Hasil/TInggi/G20T4.0R3_20260701_093412.png)
![Hasil Simulasi 4 T=4](./Hasil/TInggi/G20T4.0R4_20260701_093417.png)
![Hasil Simulasi 5 T=4](./Hasil/TInggi/G20T4.0R5_20260701_093423.png)

Konfigurasi spin akhir menunjukkan bahwa spin bernilai $+1$ dan $-1$ tersebar secara acak tanpa membentuk domain-domain besar yang seragam. Kondisi ini mengindikasikan bahwa sistem berada pada fase tidak teratur (*disordered phase*), yang merupakan karakteristik model Ising pada temperatur tinggi. Pada temperatur ini, energi termal yang dimiliki sistem cukup besar sehingga mampu mengatasi interaksi antarspin yang cenderung menyelaraskan orientasi spin.

Grafik magnetisasi menunjukkan bahwa nilai magnetisasi rata-rata terus berfluktuasi di sekitar nol selama simulasi berlangsung. Nilai rata-rata magnetisasi absolut ($|M|$) yang diperoleh dari kelima simulasi berada pada kisaran 0,001–0,016. Nilai tersebut menunjukkan bahwa jumlah spin yang berorientasi ke atas dan ke bawah hampir seimbang sehingga sistem tidak memiliki magnetisasi bersih (*net magnetization*). Fluktuasi magnetisasi yang tetap terjadi selama simulasi merupakan akibat dari pengaruh energi termal yang menyebabkan spin terus mengalami pembalikan secara acak.

Sementara itu, grafik energi memperlihatkan bahwa energi per spin berfluktuasi di sekitar nilai rata-rata sekitar $-0,55$ hingga $-0,56$. Setelah beberapa langkah Monte Carlo pertama, energi sistem mencapai keadaan yang relatif stabil dan hanya mengalami fluktuasi kecil di sekitar nilai kesetimbangannya. Hal ini menunjukkan bahwa sistem telah mencapai kesetimbangan termodinamika (*thermal equilibrium*), sehingga perubahan konfigurasi spin berikutnya hanya menghasilkan fluktuasi energi akibat proses acak yang dikendalikan oleh distribusi Boltzmann.

Nilai suseptibilitas magnetik ($\chi$) yang diperoleh berada pada kisaran 0,002–0,003, sedangkan kapasitas kalor ($C$) bernilai sangat kecil, yaitu mendekati nol. Nilai tersebut menunjukkan bahwa pada temperatur $T = 4,0$ respons sistem terhadap perubahan temperatur maupun medan magnet relatif kecil. Selain itu, kemiripan nilai rata-rata magnetisasi, energi, suseptibilitas, dan kapasitas kalor pada kelima pengulangan menunjukkan bahwa algoritma Metropolis menghasilkan hasil simulasi yang konsisten meskipun konfigurasi mikroskopis akhir berbeda pada setiap proses simulasi.

Secara keseluruhan, hasil simulasi sesuai dengan karakteristik model Ising dua dimensi pada temperatur tinggi, yaitu sistem berada pada fase paramagnetik dengan orientasi spin yang acak, magnetisasi rata-rata mendekati nol, serta energi sistem telah mencapai keadaan setimbang dan hanya mengalami fluktuasi akibat pengaruh energi termal. Temperatur yang digunakan pada simulasi, yaitu $T = 4,0$, berada di atas temperatur kritis model Ising dua dimensi ($T_c \approx 2,269$), sehingga perilaku sistem yang ditunjukkan berupa fase paramagnetik telah sesuai dengan teori.


## _Source Code_

Memasukkan Library, numpy untuk perhitungan, matplotlib untuk plot gambar, random untuk memunculkan angka random, tkinter untuk GUI di python

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import random
import tkinter as tk
from tkinter import messagebox, ttk
from datetime import datetime

Fungsi untuk menghitung nilai energi awal dengan Persamaan:
$E = -S_{ij}\sum_{n \in N(i,j)} S_n$

In [ ]:

def energiawal(grid):
    N = grid.shape[0]
    total_energi = 0
    for i in range(N): 
        for j in range(N): 
            S = grid[i,j]
            s_neighbors = (
                grid[(i+1)%N, j] + grid[(i-1)%N, j] +
                grid[i, (j+1)%N] + grid[i, (j-1)%N]
            )
            total_energi += -S*s_neighbors
    return total_energi/2


 ### Algoritma Metropolis
 
 Fungsi `metropolis_step()` adalah implementasi algoritma **Metropolis Monte Carlo** yang digunakan untuk memperbarui konfigurasi spin pada model Ising. Pada setiap iterasi, satu spin dipilih secara acak, kemudian dihitung perubahan energi akibat pembalikan spin berdasarkan interaksi dengan empat spin tetangga terdekat menggunakan **Periodic Boundary Condition (PBC)**. Perubahan energi dihitung menggunakan persamaan

$
\Delta E = 2S_{ij}\sum_{n \in N(i,j)} S_n
$

dengan:
- $(S_{ij})$ adalah nilai spin pada posisi $((i,j))$,
- $(N(i,j))$ adalah himpunan empat tetangga terdekat dari spin $((i,j))$,
- $(\sum S_n)$ adalah jumlah nilai spin tetangga.

Selanjutnya, pembalikan spin diterima apabila menghasilkan penurunan energi ($(\Delta E < 0)$) atau memenuhi probabilitas Boltzmann

$
P = e^{-\Delta E/T},
$

apabila $(\Delta E > 0)$, dengan $(T)$ merupakan temperatur sistem. Mekanisme ini memungkinkan sistem berevolusi menuju keadaan kesetimbangan termodinamika sesuai temperatur yang diberikan.

In [ ]:
def metropolis_step(grid, T):
    N = grid.shape[0]
    x, y = random.randint(0, N-1), random.randint(0, N-1)
    s_neighbors = (
        grid[(x+1)%N, y] + grid[(x-1)%N, y] +
        grid[x, (y+1)%N] + grid[x, (y-1)%N]
    )
    dE_acceptance = 0
    delta_E = 2 * grid[x, y] * s_neighbors
    if delta_E < 0 or random.random() < np.exp(-delta_E /T):
        grid[x, y] *= -1
        dE_acceptance = delta_E
    return grid, dE_acceptance

### Simulasi Model Ising

Fungsi `run_simulation()` digunakan untuk menjalankan simulasi model Ising menggunakan algoritma Metropolis Monte Carlo. Simulasi diawali dengan membangkitkan konfigurasi spin awal secara acak pada kisi berukuran $N \times N$, kemudian menghitung energi awal sistem menggunakan fungsi `energiawal()`.

Selanjutnya, selama sebanyak `n_steps` iterasi, fungsi `metropolis_step()` dipanggil untuk memperbarui konfigurasi spin berdasarkan kriteria Metropolis. Perubahan energi yang diterima ($\Delta E$) kemudian digunakan untuk memperbarui energi total sistem, yang dinyatakan sebagai

$$
E_{\text{sistem}} = E_{\text{sistem}} + \Delta E.
$$

Setiap 100 iterasi dilakukan perhitungan magnetisasi rata-rata dan energi per spin. Magnetisasi dihitung menggunakan persamaan

$$
M = \frac{1}{N^2}\sum_{i=1}^{N}\sum_{j=1}^{N} S_{ij},
$$

sedangkan energi per spin dihitung dengan

$$
E_{\text{spin}} = \frac{E_{\text{sistem}}}{N^2}.
$$

Nilai magnetisasi dan energi per spin kemudian disimpan pada setiap interval pengamatan untuk dianalisis sebagai fungsi temperatur maupun jumlah iterasi. Pada akhir simulasi, fungsi mengembalikan konfigurasi spin akhir, riwayat magnetisasi, dan riwayat energi per spin.

In [ ]:
def run_simulation(N=20, temp=1.0, n_steps=100000):
    grid = np.random.choice([-1, 1], size=(N, N))
    magnetization_history = []
    Elok_history = []
    energisistem = energiawal(grid)
    for step in range(n_steps):
        grid, dE_accept = metropolis_step(grid, temp)
        energisistem += dE_accept

        if step % 100 == 0:
            magnetization = np.mean(grid)
            magnetization_history.append(magnetization)
            Elok_history.append(energisistem/(N**2))

    return grid, magnetization_history, Elok_history

### Visualisasi Hasil Simulasi

Fungsi `plot()` digunakan untuk menjalankan simulasi model Ising sekaligus memvisualisasikan hasilnya. Fungsi ini memanggil `run_simulation()` untuk memperoleh konfigurasi spin akhir, riwayat magnetisasi, dan riwayat energi per spin pada temperatur yang ditentukan. Hasil simulasi kemudian ditampilkan dalam tiga grafik, yaitu konfigurasi akhir spin, perubahan magnetisasi terhadap langkah Monte Carlo, dan perubahan energi per spin terhadap langkah Monte Carlo.

Selain menampilkan grafik, fungsi ini juga menghitung nilai rata-rata magnetisasi, rata-rata energi per spin, suseptibilitas magnetik (\(\chi\)), dan kapasitas kalor (\(C\)). Suseptibilitas dihitung berdasarkan fluktuasi magnetisasi menggunakan persamaan

$$
\chi = \frac{\mathrm{Var}(M)}{T},
$$

sedangkan kapasitas kalor dihitung berdasarkan fluktuasi energi menggunakan persamaan

$$
C = \frac{\mathrm{Var}(E)}{T^2},
$$

dengan $\mathrm{Var}(M)$ merupakan variansi magnetisasi, $\mathrm{Var}(E)$ merupakan variansi energi per spin, dan $T$ adalah temperatur sistem.

Nilai-nilai tersebut ditampilkan sebagai informasi tambahan pada legenda grafik untuk memudahkan analisis karakteristik sistem pada temperatur tertentu. Selanjutnya, hasil visualisasi disimpan secara otomatis dalam format `.png` ke folder `hasil` ketika jendela grafik ditutup. Nama file dibuat secara otomatis berdasarkan ukuran kisi, temperatur, nomor simulasi, serta waktu penyimpanan sehingga setiap hasil simulasi memiliki nama yang unik dan tidak saling menimpa.

In [ ]:
def plot(grid_size, temperatures, monte_carlo_steps, run_index=1):
    fig, axes = plt.subplots(1, 3, figsize=(15, 8))
    fig.suptitle('2D Ising Model Simulation via Metropolis Algorithm')
    final_grid, M_history, E_history = run_simulation(N=grid_size, temp=temperatures, n_steps=monte_carlo_steps)

    ax_grid = axes[0]
    ax_grid.imshow(final_grid, cmap='binary', vmin=-1, vmax=1)
    ax_grid.set_title(f"Final State at T = {temperatures:.2f}")

    ax_mag = axes[1]
    ax_mag.plot(M_history)
    ax_mag.set_title(f"Magnetization at T = {temperatures:.2f}")
    ax_mag.set_xlabel("Monte Carlo Steps (x100)")
    ax_mag.set_ylabel("Average Magnetization")
    ax_mag.set_ylim(-1.1, 1.1)

    ax_e = axes[2]
    ax_e.plot(E_history)
    ax_e.set_title(f"Energy at T = {temperatures:.2f}")
    ax_e.set_xlabel("Monte Carlo Steps (x100)")
    ax_e.set_ylabel("Average local Energy")
    teks_legenda = mpatches.Rectangle((0, 0), 1, 1, facecolor='none', edgecolor='none')
    sus = np.var((np.array(M_history)))/temperatures
    kap = np.var((np.array(E_history)))/(temperatures**2)

    plt.legend([teks_legenda], [f'Rata-rata |M| =  {abs(np.mean(M_history)):.3f}\n Rata-rata E_lok = {abs(np.mean(E_history)):.3f}\n E_lok last state = {E_history[-1]:.3f} \n x = {sus:.3f}, C = {kap:.3f}'], handlelength=0)
    plt.tight_layout()
    
    # Otomatis buat folder 'hasil' jika belum ada
    output_dir = "hasil"
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        
    # Ambil waktu saat ini dengan format: TAHUN-BULAN-HARI_JAM-MENIT-DETIK
    waktu_sekarang = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"{output_dir}/G{grid_size}T{temperatures}R{run_index}_{waktu_sekarang}.png"
    
    # --- FITUR AUTO SAVE SAAT JENDELA DITUTUP ---
    def on_close(event):
        # Menyimpan gambar berdasarkan kondisi terakhir di layar sebelum jendela benar-benar tertutup
        fig.savefig(filename)
        print(f"Grafik perubahan berhasil disimpan otomatis ke: {filename}")

    # Menghubungkan event penutupan jendela window ke fungsi on_close
    fig.canvas.mpl_connect('close_event', on_close)
    
    # Tampilkan grafik ke pengguna (Anda bebas zoom/pan di sini)
    plt.show()
    
    return M_history, E_history

### Menjalankan Simulasi

Fungsi `jalankan_simulasi()` berfungsi sebagai penghubung antara antarmuka pengguna (GUI) dengan proses simulasi model Ising. Fungsi ini mengambil nilai parameter yang dimasukkan pengguna, yaitu ukuran kisi (`grid_size`), temperatur (`temperatures`), dan jumlah langkah Monte Carlo (`monte_carlo_steps`). Selanjutnya, fungsi memeriksa opsi jumlah simulasi yang dipilih.

Apabila pengguna memilih satu kali simulasi, fungsi `plot()` dipanggil satu kali untuk menjalankan simulasi dan menampilkan hasil visualisasi. Sebaliknya, jika pengguna memilih simulasi berulang, fungsi `plot()` dijalankan sebanyak jumlah iterasi yang ditentukan sehingga menghasilkan beberapa hasil simulasi dengan parameter yang sama.

Setelah seluruh proses simulasi selesai, program menampilkan kotak dialog (`messagebox`) sebagai notifikasi bahwa simulasi berhasil dijalankan dan hasil visualisasi telah disimpan pada folder `hasil`. Selain itu, fungsi ini menerapkan mekanisme penanganan kesalahan (*exception handling*) menggunakan `try-except` untuk memastikan bahwa seluruh parameter masukan berupa nilai numerik. Jika pengguna memasukkan data yang tidak valid, program akan menampilkan pesan kesalahan sehingga simulasi tidak dijalankan.

In [ ]:
def jalankan_simulasi():
    try:
        # Ambil nilai dari input GUI
        grid_size = int(entry_grid.get())
        temperatures = float(entry_temp.get())
        monte_carlo_steps = int(entry_steps.get())
        
        opsi_iterasi = var_opsi.get()
        
        if opsi_iterasi == "1":
            plot(grid_size, temperatures, monte_carlo_steps, run_index=1)
            messagebox.showinfo("Sukses", "Simulasi selesai! Plot disimpan di folder 'hasil'.")
        else:
            jumlah_iterasi = int(entry_iterasi.get())
            for i in range(jumlah_iterasi):
                plot(grid_size, temperatures, monte_carlo_steps, run_index=i+1)
            messagebox.showinfo("Sukses", f"Simulasi sebanyak {jumlah_iterasi} kali selesai! Semua plot disimpan di folder 'hasil'.")
            
    except ValueError:
        messagebox.showerror("Error", "Pastikan semua input diisi dengan angka yang benar!")


### Pengaturan Input Jumlah Iterasi

Fungsi `toggle_input_iterasi()` digunakan untuk mengatur status kolom masukan jumlah iterasi pada antarmuka pengguna (GUI). Fungsi ini akan memeriksa pilihan yang dipilih melalui *radio button*. Jika pengguna memilih opsi simulasi berulang, maka kolom input jumlah iterasi akan diaktifkan sehingga pengguna dapat memasukkan banyaknya pengulangan simulasi. Sebaliknya, apabila pengguna memilih simulasi satu kali, kolom tersebut akan dinonaktifkan karena tidak diperlukan. Mekanisme ini membuat antarmuka menjadi lebih interaktif, sederhana, serta mengurangi kemungkinan kesalahan dalam memasukkan parameter simulasi.

In [ ]:
def toggle_input_iterasi():
    # Mengaktifkan/menonaktifkan input jumlah iterasi berdasarkan pilihan radio button
    if var_opsi.get() == "2":
        entry_iterasi.config(state="normal")
    else:
        entry_iterasi.config(state="disabled")


### Antarmuka Pengguna (Graphical User Interface)

Bagian program ini berfungsi untuk membangun antarmuka pengguna (*Graphical User Interface* atau GUI) menggunakan pustaka **Tkinter**. Antarmuka dirancang agar pengguna dapat menjalankan simulasi model Ising tanpa perlu mengubah kode program secara langsung. GUI terdiri atas beberapa komponen utama, yaitu kolom masukan ukuran kisi (*Grid Size*), temperatur (*Temperature*), dan jumlah langkah Monte Carlo (*Monte Carlo Steps*), yang masing-masing telah diberikan nilai awal (*default value*) untuk memudahkan penggunaan.

Selain itu, tersedia pilihan untuk menjalankan simulasi sebanyak satu kali atau beberapa kali melalui *radio button*. Apabila opsi simulasi berulang dipilih, kolom masukan jumlah iterasi akan diaktifkan sehingga pengguna dapat menentukan banyaknya pengulangan simulasi. Sebaliknya, pada simulasi satu kali, kolom tersebut dinonaktifkan agar tidak terjadi kesalahan dalam pengisian parameter.

Pada bagian bawah antarmuka terdapat tombol **Mulai Simulasi** yang akan memanggil fungsi `jalankan_simulasi()` untuk menjalankan proses simulasi sesuai parameter yang telah dimasukkan. Seluruh komponen antarmuka disusun menggunakan kombinasi `Frame`, `Label`, `Entry`, `Radiobutton`, dan `Button` sehingga tampilan menjadi lebih terstruktur, mudah dipahami, dan nyaman digunakan.

In [ ]:
# --- DESAIN INTERFACES GUI ---
root = tk.Tk()
root.title("Simulasi 2D Ising Model")
root.geometry("400x380")
root.resizable(False, False)

# Mengatur bobot kolom di root agar frame bisa melebar jika window membesar
root.columnconfigure(0, weight=1)

# Frame Input Utama
frame_input = ttk.LabelFrame(root, text=" Parameter Simulasi ", padding=15)
frame_input.pack(fill="x", padx=15, pady=10)

# Mengatur bobot kolom di dalam frame_input agar kolom entry (kolom 1) bisa melebar
frame_input.columnconfigure(1, weight=1)

# Input Grid Size
ttk.Label(frame_input, text="Grid Size:").grid(row=0, column=0, sticky="w", pady=5)
entry_grid = ttk.Entry(frame_input)
entry_grid.insert(0, "20") # nilai default
# PERBAIKAN: Mengganti fill dan expand dengan sticky="ew"
entry_grid.grid(row=0, column=1, sticky="ew", pady=5)

# Input Temperature
ttk.Label(frame_input, text="Temperature (T):").grid(row=1, column=0, sticky="w", pady=5)
entry_temp = ttk.Entry(frame_input)
entry_temp.insert(0, "1.0")
# PERBAIKAN: Mengganti fill dan expand dengan sticky="ew"
entry_temp.grid(row=1, column=1, sticky="ew", pady=5)

# Input Monte Carlo Steps
ttk.Label(frame_input, text="Monte Carlo Steps:").grid(row=2, column=0, sticky="w", pady=5)
entry_steps = ttk.Entry(frame_input)
entry_steps.insert(0, "100000")
# PERBAIKAN: Mengganti fill dan expand dengan sticky="ew"
entry_steps.grid(row=2, column=1, sticky="ew", pady=5)

# Frame Pilihan Iterasi
frame_opsi = ttk.LabelFrame(root, text=" Opsi Iterasi ", padding=15)
frame_opsi.pack(fill="x", padx=15, pady=5)

var_opsi = tk.StringVar(value="1")

r1 = ttk.Radiobutton(frame_opsi, text="Jalankan 1 Kali Semisal", value="1", variable=var_opsi, command=toggle_input_iterasi)
r1.grid(row=0, column=0, columnspan=2, sticky="w", pady=2)

r2 = ttk.Radiobutton(frame_opsi, text="Ulangi dengan jumlah iterasi:", value="2", variable=var_opsi, command=toggle_input_iterasi)
r2.grid(row=1, column=0, sticky="w", pady=2)

entry_iterasi = ttk.Entry(frame_opsi, width=10, state="disabled")
entry_iterasi.insert(0, "2")
entry_iterasi.grid(row=1, column=1, sticky="w", padx=5, pady=2)

# Tombol Run
btn_run = ttk.Button(root, text="MULAI SIMULASI", command=jalankan_simulasi)
btn_run.pack(fill="x", padx=15, pady=20)

root.mainloop()


Berikut _Full Code_ nya : 

import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import random
import tkinter as tk
from tkinter import messagebox, ttk
from datetime import datetime

def energiawal(grid):
    N = grid.shape[0]
    total_energi = 0
    for i in range(N): 
        for j in range(N): 
            S = grid[i,j]
            s_neighbors = (
                grid[(i+1)%N, j] + grid[(i-1)%N, j] +
                grid[i, (j+1)%N] + grid[i, (j-1)%N]
            )
            total_energi += -S*s_neighbors
    return total_energi/2

def metropolis_step(grid, T):
    N = grid.shape[0]
    x, y = random.randint(0, N-1), random.randint(0, N-1)
    s_neighbors = (
        grid[(x+1)%N, y] + grid[(x-1)%N, y] +
        grid[x, (y+1)%N] + grid[x, (y-1)%N]
    )
    dE_acceptance = 0
    delta_E = 2 * grid[x, y] * s_neighbors
    if delta_E < 0 or random.random() < np.exp(-delta_E /T):
        grid[x, y] *= -1
        dE_acceptance = delta_E
    return grid, dE_acceptance

def run_simulation(N=20, temp=1.0, n_steps=100000):
    grid = np.random.choice([-1, 1], size=(N, N))
    magnetization_history = []
    Elok_history = []
    energisistem = energiawal(grid)
    for step in range(n_steps):
        grid, dE_accept = metropolis_step(grid, temp)
        energisistem += dE_accept

        if step % 100 == 0:
            magnetization = np.mean(grid)
            magnetization_history.append(magnetization)
            Elok_history.append(energisistem/(N**2))

    return grid, magnetization_history, Elok_history

def plot(grid_size, temperatures, monte_carlo_steps, run_index=1):
    fig, axes = plt.subplots(1, 3, figsize=(15, 8))
    fig.suptitle('2D Ising Model Simulation via Metropolis Algorithm')
    final_grid, M_history, E_history = run_simulation(N=grid_size, temp=temperatures, n_steps=monte_carlo_steps)

    ax_grid = axes[0]
    ax_grid.imshow(final_grid, cmap='binary', vmin=-1, vmax=1)
    ax_grid.set_title(f"Final State at T = {temperatures:.2f}")

    ax_mag = axes[1]
    ax_mag.plot(M_history)
    ax_mag.set_title(f"Magnetization at T = {temperatures:.2f}")
    ax_mag.set_xlabel("Monte Carlo Steps (x100)")
    ax_mag.set_ylabel("Average Magnetization")
    ax_mag.set_ylim(-1.1, 1.1)

    ax_e = axes[2]
    ax_e.plot(E_history)
    ax_e.set_title(f"Energy at T = {temperatures:.2f}")
    ax_e.set_xlabel("Monte Carlo Steps (x100)")
    ax_e.set_ylabel("Average local Energy")
    teks_legenda = mpatches.Rectangle((0, 0), 1, 1, facecolor='none', edgecolor='none')
    sus = np.var((np.array(M_history)))/temperatures
    kap = np.var((np.array(E_history)))/(temperatures**2)

    plt.legend([teks_legenda], [f'Rata-rata |M| =  {abs(np.mean(M_history)):.3f}\n Rata-rata E_lok = {abs(np.mean(E_history)):.3f}\n E_lok last state = {E_history[-1]:.3f} \n x = {sus:.3f}, C = {kap:.3f}'], handlelength=0)
    plt.tight_layout()
    
    # Otomatis buat folder 'hasil' jika belum ada
    output_dir = "hasil"
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        
    # Ambil waktu saat ini dengan format: TAHUN-BULAN-HARI_JAM-MENIT-DETIK
    waktu_sekarang = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"{output_dir}/G{grid_size}T{temperatures}R{run_index}_{waktu_sekarang}.png"
    
    # --- FITUR AUTO SAVE SAAT JENDELA DITUTUP ---
    def on_close(event):
        # Menyimpan gambar berdasarkan kondisi terakhir di layar sebelum jendela benar-benar tertutup
        fig.savefig(filename)
        print(f"Grafik perubahan berhasil disimpan otomatis ke: {filename}")

    # Menghubungkan event penutupan jendela window ke fungsi on_close
    fig.canvas.mpl_connect('close_event', on_close)
    
    # Tampilkan grafik ke pengguna (Anda bebas zoom/pan di sini)
    plt.show()
    
    return M_history, E_history


# --- FUNGSI LOGIKA GUI ---
def jalankan_simulasi():
    try:
        # Ambil nilai dari input GUI
        grid_size = int(entry_grid.get())
        temperatures = float(entry_temp.get())
        monte_carlo_steps = int(entry_steps.get())
        
        opsi_iterasi = var_opsi.get()
        
        if opsi_iterasi == "1":
            plot(grid_size, temperatures, monte_carlo_steps, run_index=1)
            messagebox.showinfo("Sukses", "Simulasi selesai! Plot disimpan di folder 'hasil'.")
        else:
            jumlah_iterasi = int(entry_iterasi.get())
            for i in range(jumlah_iterasi):
                plot(grid_size, temperatures, monte_carlo_steps, run_index=i+1)
            messagebox.showinfo("Sukses", f"Simulasi sebanyak {jumlah_iterasi} kali selesai! Semua plot disimpan di folder 'hasil'.")
            
    except ValueError:
        messagebox.showerror("Error", "Pastikan semua input diisi dengan angka yang benar!")

def toggle_input_iterasi():
    # Mengaktifkan/menonaktifkan input jumlah iterasi berdasarkan pilihan radio button
    if var_opsi.get() == "2":
        entry_iterasi.config(state="normal")
    else:
        entry_iterasi.config(state="disabled")

# --- DESAIN INTERFACES GUI ---
root = tk.Tk()
root.title("Simulasi 2D Ising Model")
root.geometry("400x380")
root.resizable(False, False)

# Mengatur bobot kolom di root agar frame bisa melebar jika window membesar
root.columnconfigure(0, weight=1)

# Frame Input Utama
frame_input = ttk.LabelFrame(root, text=" Parameter Simulasi ", padding=15)
frame_input.pack(fill="x", padx=15, pady=10)

# Mengatur bobot kolom di dalam frame_input agar kolom entry (kolom 1) bisa melebar
frame_input.columnconfigure(1, weight=1)

# Input Grid Size
ttk.Label(frame_input, text="Grid Size:").grid(row=0, column=0, sticky="w", pady=5)
entry_grid = ttk.Entry(frame_input)
entry_grid.insert(0, "20") # nilai default
# PERBAIKAN: Mengganti fill dan expand dengan sticky="ew"
entry_grid.grid(row=0, column=1, sticky="ew", pady=5)

# Input Temperature
ttk.Label(frame_input, text="Temperature (T):").grid(row=1, column=0, sticky="w", pady=5)
entry_temp = ttk.Entry(frame_input)
entry_temp.insert(0, "1.0")
# PERBAIKAN: Mengganti fill dan expand dengan sticky="ew"
entry_temp.grid(row=1, column=1, sticky="ew", pady=5)

# Input Monte Carlo Steps
ttk.Label(frame_input, text="Monte Carlo Steps:").grid(row=2, column=0, sticky="w", pady=5)
entry_steps = ttk.Entry(frame_input)
entry_steps.insert(0, "100000")
# PERBAIKAN: Mengganti fill dan expand dengan sticky="ew"
entry_steps.grid(row=2, column=1, sticky="ew", pady=5)

# Frame Pilihan Iterasi
frame_opsi = ttk.LabelFrame(root, text=" Opsi Iterasi ", padding=15)
frame_opsi.pack(fill="x", padx=15, pady=5)

var_opsi = tk.StringVar(value="1")

r1 = ttk.Radiobutton(frame_opsi, text="Jalankan 1 Kali Semisal", value="1", variable=var_opsi, command=toggle_input_iterasi)
r1.grid(row=0, column=0, columnspan=2, sticky="w", pady=2)

r2 = ttk.Radiobutton(frame_opsi, text="Ulangi dengan jumlah iterasi:", value="2", variable=var_opsi, command=toggle_input_iterasi)
r2.grid(row=1, column=0, sticky="w", pady=2)

entry_iterasi = ttk.Entry(frame_opsi, width=10, state="disabled")
entry_iterasi.insert(0, "2")
entry_iterasi.grid(row=1, column=1, sticky="w", padx=5, pady=2)

# Tombol Run
btn_run = ttk.Button(root, text="MULAI SIMULASI", command=jalankan_simulasi)
btn_run.pack(fill="x", padx=15, pady=20)

root.mainloop()
